#### Initialize

In [1]:
import sys
from pathlib import Path
import pandas as pd
REPROCESS = True

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
DATA_BUCKET = PARENT / "server/scripts/adaptive_hexsearch/out/places"
SEED_LEDGER = PARENT / "server/scripts/adaptive_hexsearch/map/seed_ledger.csv"

ARCHIVE = DATA_BUCKET.parent / "places-archived"
if not ARCHIVE.exists(): ARCHIVE.mkdir(parents=True, exist_ok=True)
PROCESSED = DATA_BUCKET.parent / "places-processed"
if not PROCESSED.exists(): PROCESSED.mkdir(parents=True, exist_ok=True)
if REPROCESS:
    RAWS = [f for f in ARCHIVE.rglob("*.csv") if f.is_file()]
else:
    RAWS = [f for f in DATA_BUCKET.glob("*.csv") if f.is_file()]

#### Merge

In [2]:
SEED_DF = pd.read_csv(SEED_LEDGER)
raw_dfs = []
for f in RAWS:

    try: df = pd.read_csv(f)
    except: df = pd.DataFrame()
    if "dineIn" in df.columns.to_list(): df.drop(columns=["dineIn"], inplace=True)
    if "regularOpeningHours" in df.columns.to_list(): df.drop(columns=["regularOpeningHours"], inplace=True)
    if "openingDate" in df.columns.to_list(): df.drop(columns=["openingDate"], inplace=True)

    path_ids = f.stem.split("-")
    df["sourceID"] = f.stem
    df["seedID"] = path_ids[0]
    df["Level"] = len(path_ids) - 1

    raw_dfs.append(df)

raw_df = pd.concat(raw_dfs, ignore_index=True)
# Drop Duplicates
raw_df.drop_duplicates(subset=["id"], inplace=True)
# Check Validity
df_validity = pd.DataFrame({ col: raw_df[col].notna().sum() / len(raw_df) for col in raw_df.columns }, index=[0])

# Drop Missing Ratings
df_level1 = raw_df[raw_df['rating'].notna() & raw_df['userRatingCount'].notna()]
print(f"""
    Dropped {len(raw_df) - len(df_level1)} Rows; 
    Dropped No Ratings Zones{set(raw_df["seedID"].unique()) - set(df_level1["seedID"].unique())}:
""")
display(df_validity)

# Sort by ID and Reset Index
df_level1.sort_values(by="id", inplace=True)
df_level1.reset_index(drop=True, inplace=True)


    Dropped 46 Rows; 
    Dropped No Ratings Zones{'236'}:



,id,displayName,primaryTypeDisplayName,rating,userRatingCount,location,shortFormattedAddress,googleMapsUri,priceRange,priceLevel,websiteUri,businessStatus,types,primaryType,sourceID,seedID,Level
0,1.0,1.0,1.0,0.895455,0.895455,1.0,1.0,1.0,0.759091,0.293182,0.688636,1.0,1.0,1.0,1.0,1.0,1.0


#### PARSE

In [3]:
from server.scripts.clean_data.type_parsing import parse_type, check_takeaway, predict_cuisine_from_name

df_level2 = df_level1.copy()
df_level2["predictedType"] = df_level2.apply(predict_cuisine_from_name, axis=1)
df_level2["cuisineType"] = df_level2.apply(parse_type, axis=1)
df_level2["venueType"] = df_level2.apply(check_takeaway, axis=1)

# ── Diagnostics ───────────────────────────────────────────────────────────────
dist = df_level2["cuisineType"].value_counts()
unresolved = (df_level2["cuisineType"] == "Unspecified").sum()
print(f"Unique cuisineTypes : {dist.nunique()}")
print(f"Still 'Unspecified'  : {unresolved} / {len(df_level2)}  ({unresolved/len(df_level2):.1%})")
df_level2[df_level2["cuisineType"]=="Unspecified"].head(2)

Unique cuisineTypes : 21
Still 'Unspecified'  : 40 / 394  (10.2%)


,id,displayName,primaryTypeDisplayName,rating,userRatingCount,location,shortFormattedAddress,googleMapsUri,priceRange,priceLevel,websiteUri,businessStatus,types,primaryType,sourceID,seedID,Level,predictedType,cuisineType,venueType
16,ChIJ1XBtbysFdkgRVVNKlINSotw,Zen-G Kitchen,Restaurant,5.0,2.0,"{'latitude': 51.474875499999996, 'longitude': ...","Lindford Street Business Estate, 76 Stewart's ...",https://maps.google.com/?cid=15898360359653364...,NaN,NaN,https://zengkitchen.co.uk/,OPERATIONAL,"['restaurant', 'point_of_interest', 'food', 'e...",restaurant,157-5-0-1-5,157,4,,Unspecified,Dine-In
22,ChIJ36APYg0FdkgR8DRF4-44s_k,Upstairs,Restaurant,4.7,102.0,"{'latitude': 51.4635207, 'longitude': -0.14152...","4 The Polygon, London",https://maps.google.com/?cid=17992787534941598...,NaN,NaN,http://www.trinity-upstairs.co.uk/,OPERATIONAL,"['restaurant', 'food', 'point_of_interest', 'e...",restaurant,183-3,183,1,,Unspecified,Dine-In


#### LOCATE

In [ ]:
import ast
import h3

GRID_REF = PARENT / "server/scripts/region_grid/h3_9.csv"
grid_ref = pd.read_csv(GRID_REF)
df_level3 = df_level2.copy()

# ── H3 version compat ────────────────────────────────────────────────────────
_H3_V4 = hasattr(h3, "latlng_to_cell")

def _latlng_to_cell(lat: float, lon: float, res: int = 9) -> str:
    return h3.latlng_to_cell(lat, lon, res) if _H3_V4 else h3.geo_to_h3(lat, lon, res)

def _grid_disk(cell: str, k: int) -> set:
    return h3.grid_disk(cell, k) if _H3_V4 else h3.k_ring(cell, k)

# ── Parse location string → (lat, lon) ───────────────────────────────────────
def _parse_latlon(loc_str) -> tuple[float | None, float | None]:
    try:
        loc = ast.literal_eval(str(loc_str))
        return loc["latitude"], loc["longitude"]
    except Exception:
        return None, None

# ── H3 level-9 host cell ─────────────────────────────────────────────────────
def _host_cell(lat, lon) -> str | None:
    if pd.isna(lat) or pd.isna(lon):
        return None
    return _latlng_to_cell(float(lat), float(lon), 9)

# ── Surrounding cells (ring 1 + ring 2, excluding host) ──────────────────────
def _neighbour_cells(host: str | None) -> list[str]:
    if host is None:
        return []
    return sorted(_grid_disk(host, 2) - {host})

# ── Apply ────────────────────────────────────────────────────────────────────
_latlon = df_level3["location"].apply(_parse_latlon)
df_level3["_lat"] = _latlon.apply(lambda t: t[0])
df_level3["_lon"] = _latlon.apply(lambda t: t[1])

df_level3["hostLocale"]      = df_level3.apply(lambda r: _host_cell(r["_lat"], r["_lon"]), axis=1)
df_level3["neighbourLocale"] = df_level3["hostLocale"].apply(_neighbour_cells)

df_level3.drop(columns=["_lat", "_lon"], inplace=True)

# ── Diagnostics ───────────────────────────────────────────────────────────────
missing = df_level3["hostLocale"].isna().sum()
print(f"Restaurants assigned a hostLocale : {len(df_level3) - missing} / {len(df_level3)}")
print(f"Missing hostLocale                : {missing}")
print(f"Unique host cells                 : {df_level3['hostLocale'].nunique()}")
print(f"Neighbour list length sample      : {df_level3['neighbourLocale'].iloc[0].__len__()} cells")
df_level3[["displayName", "hostLocale", "neighbourLocale"]].head(3)

,tileID,boundary,center_lat,center_lon
0,89194ad0003ffff,"[[51.46014332971915, -0.04859045289981933], [5...",51.458396,-0.048160
1,89194ad0007ffff,"[[51.45762777059396, -0.045639633896778734], [...",51.455880,-0.045209
2,89194ad000bffff,"[[51.459931894144546, -0.05320053551237643], [...",51.458184,-0.052770
3,89194ad000fffff,"[[51.45741639871367, -0.0502494073392683], [51...",51.455669,-0.049819
4,89194ad0013ffff,"[[51.46287013250036, -0.04693136398562476], [5...",51.461123,-0.046501
...,...,...,...,...
4188,89195da6e2bffff,"[[51.57834497754931, -0.12921033151808747], [5...",51.576599,-0.128776
4189,89195da6e2fffff,"[[51.575831713005215, -0.12624779381275164], [...",51.574086,-0.125814
4190,89195da6e33ffff,"[[51.58128194147675, -0.12292795438991982], [5...",51.579536,-0.122494
4191,89195da6e37ffff,"[[51.57876861516809, -0.11996570065021293], [5...",51.577022,-0.119532


#### Export

In [4]:
from datetime import datetime

for seed_id, group in raw_df.groupby("seedID"):
    save_path = PROCESSED / f"{seed_id}.csv"
    if not save_path.parent.exists():
        save_path.parent.mkdir(parents=True, exist_ok=True)
    group.to_csv(save_path, index=False)

if not REPROCESS:
    for f in RAWS:
        # Move raw file to archive
        archive_path = ARCHIVE / f"{datetime.now().strftime('%Y-%m-%d')}/{f.name}"
        if not archive_path.parent.exists():
            archive_path.parent.mkdir(parents=True, exist_ok=True)
        f.rename(archive_path)